# Taller 03 - Consumo de API + Power BI

**API:** Open Library (https://openlibrary.org/developers/api)  
**Objetivo:** Extraer datos de libros por categoría, transformarlos y exportarlos para análisis en Power BI

In [1]:
import requests
import pandas as pd
import numpy as np

## Parte 1 - Consulta a la API

In [2]:
# verifico que la API responda antes de hacer todas las consultas
r = requests.get('https://openlibrary.org/search.json?subject=science&limit=1')
print('Status code:', r.status_code)
print('API disponible:', r.status_code == 200)

Status code: 200
API disponible: True


In [3]:
# consulto 5 categorias, 100 libros cada una
# los campos que me interesan: titulo, autor, año, ediciones, ratings, idioma
categorias = ['science', 'history', 'fiction', 'technology', 'art']
campos = 'title,author_name,first_publish_year,edition_count,ratings_average,ratings_count,language'

libros = []

for cat in categorias:
    url = f'https://openlibrary.org/search.json?subject={cat}&limit=100&fields={campos}'
    response = requests.get(url)
    data = response.json()

    for doc in data.get('docs', []):
        doc['categoria'] = cat
        libros.append(doc)

    print(f'{cat}: {len(data.get("docs", []))} libros obtenidos')

print(f'\nTotal registros: {len(libros)}')

science: 100 libros obtenidos
history: 100 libros obtenidos
fiction: 100 libros obtenidos
technology: 100 libros obtenidos
art: 100 libros obtenidos

Total registros: 500


In [4]:
# convierto a DataFrame
df = pd.DataFrame(libros)
df.shape

(500, 8)

In [5]:
df.head(3)

,author_name,edition_count,first_publish_year,language,title,ratings_average,ratings_count,categoria
0,[Neil Alexander Campbell],57,1987,"[eng, spa, fre]",Biology,3.880000,25.0,science
1,[Dan Simmons],29,1990,"[pol, por, fre, eng, ita, spa]",The Fall of Hyperion,4.053571,56.0,science
2,[刘慈欣],27,2010,"[chi, eng, fre, spa, ger]",Death's End (The Three-Body Problem Series Boo...,4.350000,100.0,science


In [6]:
# algunos campos vienen como listas (author_name, language)
df.dtypes

,0
author_name,object
edition_count,int64
first_publish_year,int64
language,object
title,object
ratings_average,float64
ratings_count,float64
categoria,object


## Parte 2 - Limpieza y transformación

In [7]:
df.isnull().sum()

,0
author_name,13
edition_count,0
first_publish_year,0
language,2
title,0
ratings_average,48
ratings_count,48
categoria,0


In [8]:
# author_name y language vienen como listas, me quedo con el primero
df['autor'] = df['author_name'].apply(lambda x: x[0] if isinstance(x, list) else 'Desconocido')
df['idioma'] = df['language'].apply(lambda x: x[0] if isinstance(x, list) else 'unknown')

In [9]:
# los codigos de idioma son abreviaciones, los mapeo a nombres legibles
print(df['idioma'].value_counts().head(10))

idioma
eng    255
ger     67
spa     30
chi     20
fre     19
por     15
pol     12
ita     11
tur     10
heb     10
Name: count, dtype: int64


In [10]:
idiomas_map = {
    'eng': 'English',
    'spa': 'Spanish',
    'fre': 'French',
    'ger': 'German',
    'ita': 'Italian',
    'por': 'Portuguese',
    'rus': 'Russian',
    'chi': 'Chinese',
    'jpn': 'Japanese',
    'dut': 'Dutch'
}
df['idioma'] = df['idioma'].map(idiomas_map).fillna('Other')
df['idioma'].value_counts()

,count
idioma,
English,255
Other,70
German,67
Spanish,30
Chinese,20
French,19
Portuguese,15
Italian,11
Russian,6


In [11]:
# elimino filas sin rating, no sirven para analisis
df = df.dropna(subset=['ratings_average'])
print(f'Filas restantes: {len(df)}')

Filas restantes: 452


In [12]:
# creo columna decada para agrupar por periodo de publicacion
df['decada'] = (df['first_publish_year'] // 10 * 10).astype(int).astype(str) + 's'
df['decada'].value_counts().sort_index()

,count
decada,
0s,3
1460s,1
1490s,1
1540s,1
1600s,1
1650s,1
1700s,1
1740s,1
1750s,1


In [13]:
# renombro columnas y me quedo con las relevantes
df = df.rename(columns={
    'title': 'titulo',
    'first_publish_year': 'anio_publicacion',
    'edition_count': 'num_ediciones',
    'ratings_average': 'rating_promedio',
    'ratings_count': 'num_ratings'
})

df = df[['titulo', 'autor', 'anio_publicacion', 'decada', 'idioma', 'categoria', 'num_ediciones', 'rating_promedio', 'num_ratings']]
df.head()

,titulo,autor,anio_publicacion,decada,idioma,categoria,num_ediciones,rating_promedio,num_ratings
0,Biology,Neil Alexander Campbell,1987,1980s,English,science,57,3.880000,25.0
1,The Fall of Hyperion,Dan Simmons,1990,1990s,Other,science,29,4.053571,56.0
2,Death's End (The Three-Body Problem Series Boo...,刘慈欣,2010,2010s,Chinese,science,27,4.350000,100.0
3,Понедельник начинается в субботу; Пикник на об...,Аркадий Стругацкий,1978,1970s,French,science,23,4.300000,70.0
4,The Caves of Steel,Isaac Asimov,1953,1950s,Other,science,61,4.307692,26.0


In [14]:
# estadisticas descriptivas del dataset final
df.describe().round(2)

,anio_publicacion,num_ediciones,rating_promedio,num_ratings
count,452.00,452.00,452.00,452.00
mean,1953.55,135.73,4.02,56.50
std,172.44,403.05,0.58,114.41
min,0.00,1.00,1.00,1.00
25%,1951.75,10.00,3.86,5.00
50%,1983.00,26.00,4.07,19.50
75%,2002.25,68.50,4.29,63.25
max,2025.00,4038.00,5.00,1170.00


In [15]:
# top 5 libros mejor calificados
df.sort_values('rating_promedio', ascending=False).head(5)[['titulo', 'autor', 'categoria', 'rating_promedio', 'num_ratings']]

,titulo,autor,categoria,rating_promedio,num_ratings
466,Imaginative Realism,James Gurney,art,5.0,3.0
452,What great paintings say,Rose-Marie Hagen,art,5.0,1.0
458,Design basics,David A. Lauer,art,5.0,1.0
459,Color and light,James Gurney,art,5.0,3.0
444,Concerning the spiritual in art,Wassily Kandinsky,art,5.0,3.0


## Parte 3 - Exportar CSV

In [16]:
df.to_csv('libros_openlibrary.csv', index=False)
print(f'CSV exportado: {len(df)} filas, {df.shape[1]} columnas')

CSV exportado: 452 filas, 9 columnas
